<a href="https://colab.research.google.com/github/nejumi/weave-initial-course/blob/main/notebooks/03_advanced/02_art_intro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 07. OpenPipe ART — エージェント強化学習の基礎

OpenPipe ART（Agent Reinforcement Trainer）を使って、
LLM エージェントを強化学習で改善する方法を学びます。

## 学習内容
1. ART の概念（GRPO / CISPO、Client-Server アーキテクチャ）
2. `TrainableModel` と `LocalBackend` のセットアップ
3. `Trajectory` の構造
4. 最小限のトレーニングループ

> **GPU 必須**: Colab の場合は **ランタイム → GPU を変更 → T4** を選択してください。


In [ ]:
# GPU 確認
import subprocess
result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                        capture_output=True, text=True)
print("GPU:", result.stdout.strip() if result.returncode == 0 else "GPU 未検出")


In [ ]:
# ART のインストール（Python >=3.11 / GPU 必須）
# Colab: ランタイム → GPU を変更 → T4 を選択してから実行
!pip install -q uv
!uv pip install --system -q \
  "openpipe-art[backend]"

In [ ]:
import os
try:
    import google.colab
    IN_COLAB = True
    from google.colab import userdata
    # WANDB_BASE_URL: 値がある場合のみセット（空 / 未設定 → SaaS デフォルト）
    _base_url = (userdata.get("WANDB_BASE_URL") or "").strip()
    if _base_url:
        os.environ["WANDB_BASE_URL"] = _base_url
    os.environ.setdefault("WANDB_API_KEY",  userdata.get("WANDB_API_KEY"))
    os.environ.setdefault("OPENAI_API_KEY", userdata.get("OPENAI_API_KEY"))
    os.environ.setdefault("WANDB_ENTITY",   userdata.get("WANDB_ENTITY"))
    os.environ.setdefault("WANDB_PROJECT",  userdata.get("WANDB_PROJECT") or "weave-course")
except ImportError:
    IN_COLAB = False
    from dotenv import load_dotenv; load_dotenv()
    # ローカル: .env に空欄で書かれていた場合も削除
    if not os.environ.get("WANDB_BASE_URL", "").strip():
        os.environ.pop("WANDB_BASE_URL", None)

ENTITY  = os.environ["WANDB_ENTITY"]
PROJECT = os.environ.get("WANDB_PROJECT", "weave-course")
_target = os.environ.get("WANDB_BASE_URL", "https://api.wandb.ai (SaaS)")
print(f"Entity : {ENTITY}")
print(f"Project: {PROJECT}")
print(f"W&B URL: {_target}")


In [ ]:
import wandb
import weave
wandb.login()
weave_client = weave.init(f"{ENTITY}/weave-course-art")


## 1. ART の概念

### 強化学習ループ

```
[Phase 1: Rollout]
  エージェントが N 個の軌跡（Trajectory）を並列実行
  → 各軌跡に報酬（reward）を付与

[Phase 2: Training]
  CISPO (GRPO 変種) でポリシー更新
  → LoRA アダプタを保存・リロード

[repeat N iterations]
```

### 主要コンポーネント

| クラス | 役割 |
|---|---|
| `TrainableModel` | 推論クライアント + チェックポイント管理 |
| `LocalBackend` | vLLM サーバー + トレーニングエンジン（GPU 上で動作） |
| `Trajectory` | メッセージ列 + 報酬 のコンテナ |
| `TrajectoryGroup` | 同一タスクの複数軌跡（GRPO の比較に使用） |


## 2. モデルとバックエンドのセットアップ

In [ ]:
import os
import art
from art.local import LocalBackend
from art import dev

# V100 / Volta (CC 7.0) 互換パッチ:
# vLLM の LoRA triton カーネルは CC<8.0 でコンパイル失敗するため、
# PunicaWrapperCPU (純粋 PyTorch 実装) に差し替える
import vllm.platforms.cuda as _cuda_platform
@classmethod
def _v100_punica(cls): return "vllm.lora.punica_wrapper.punica_cpu.PunicaWrapperCPU"
if hasattr(torch := __import__("torch"), "cuda") and torch.cuda.get_device_capability()[0] < 8:
    _cuda_platform.CudaPlatform.get_punica_wrapper = _v100_punica

# LocalBackend: in_process=True で Colab 互換（サブプロセス不要）
backend = LocalBackend(in_process=True)

model = art.TrainableModel(
    name="qwen-1.5b-tool-use",
    project="weave-course-art",
    base_model="Qwen/Qwen2.5-1.5B-Instruct",
    # CC < 8.0 (V100/Volta) では sleep mode / CUDA graph 無効化が必要
    _internal_config=dev.InternalModelConfig(
        engine_args=dev.EngineArgs(
            enable_sleep_mode=False,  # sleep mode allocator は CC<8 で再試行時にクラッシュ
            enforce_eager=True,       # CUDA graph は CC<8 で OOM を起こしやすい
        ),
        init_args=dev.InitArgs(
            gpu_memory_utilization=0.7,  # sleep mode OFF 時は 0.55 → 0.7 に増加
            max_seq_length=4096,          # 32768 は KV cache 不足になるため削減
        ),
    ) if hasattr(torch, "cuda") and torch.cuda.get_device_capability()[0] < 8 else None,
)

# バックエンドに登録（推論サーバーを起動）
await model.register(backend)
print(f"モデル準備完了: {model.inference_model_name}")
print(f"現在のステップ: {await model.get_step()}")

## 3. Trajectory の構造

`Trajectory` はマルチターンの会話と最終的な報酬を格納するコンテナです。


In [ ]:
# Trajectory の基本構造
trajectory = art.Trajectory(
    messages_and_choices=[
        {"role": "system", "content": "You are a helpful math assistant."}
    ],
    reward=0.0,
)

# ユーザーメッセージを追加
trajectory.messages_and_choices.append({"role": "user", "content": "What is 6 * 7?"})

# LLM から応答を取得
client = model.openai_client()
response = await client.chat.completions.create(
    model=model.inference_model_name,
    messages=trajectory.messages(),  # messages() で OpenAI 形式に変換
    max_completion_tokens=64,
)
trajectory.messages_and_choices.append(response.choices[0])  # Choice オブジェクトをそのまま追加

# 報酬を付与
answer_text = response.choices[0].message.content or ""
trajectory.reward = 1.0 if "42" in answer_text else 0.0

print("応答:", answer_text)
print("報酬:", trajectory.reward)
print("メッセージ数:", len(trajectory.messages()))


## 4. 最小限のトレーニングループ（動作確認用）

In [ ]:
import re

SYSTEM_PROMPT = "You are a math assistant. Answer with only the final number, no explanation."

async def single_rollout(model: art.TrainableModel, question: str, answer: int) -> art.Trajectory:
    """1 つの問題を解いて報酬を返す"""
    traj = art.Trajectory(
        messages_and_choices=[{"role": "system", "content": SYSTEM_PROMPT}],
        reward=0.0,
    )
    traj.messages_and_choices.append({"role": "user", "content": question})

    client = model.openai_client()
    resp = await client.chat.completions.create(
        model=model.inference_model_name,
        messages=traj.messages(),
        max_completion_tokens=32,
    )
    traj.messages_and_choices.append(resp.choices[0])

    output = resp.choices[0].message.content or ""
    numbers = re.findall(r"-?\d+", output.replace(",", ""))
    predicted = int(numbers[0]) if numbers else None
    traj.reward = 1.0 if predicted == answer else 0.0
    return traj

# 1 問だけ試す
traj = await single_rollout(model, "What is 13 + 29?", 42)
print(f"軌跡: reward={traj.reward}")
print(f"応答: {traj.messages()[-1]['content']}")


## まとめ

次: **03_advanced/03_art_agentic_rl.ipynb** → 完全なトレーニングループ + Weave 統合